# 14. Full Model Evaluation Sweep (All Experiments × All Models)

Runs automatic-metric evaluation (SacreBLEU, chrF++, Lexical Accuracy, and neural COMET) against **every saved checkpoint** under `checkpoints/qwen/` and `checkpoints/mistral/`, plus the E0 zero-shot baseline for both models. Unlike `09_translation_evaluation.ipynb` (which evaluates one model/experiment at a time via manually-set variables), this notebook discovers whatever has actually been trained on disk and evaluates all of it in one pass — so it naturally covers E1–E10 as those checkpoints appear, not just whatever the last manual run happened to target.

**Requires GPU.** This is expensive: it runs full generation for two directions (English→Ekegusii, Ekegusii→English) against the ~4,928-row master test split for every checkpoint found, for both models, then a further COMET pass. Adjust `MODELS` and `MAX_COMET_SAMPLES` below if you want a faster partial run.

In [ ]:
# ============================================================
# PATH & ENVIRONMENT BOOSTER — Guarantees project path setup
# ============================================================
import os, sys, site, glob

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
for conda_site in glob.glob('/opt/conda/lib/python3.*/site-packages'):
    if conda_site not in sys.path:
        sys.path.insert(0, conda_site)

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')

In [ ]:
# Which base models to sweep, and how many test sentences to use for the
# (slower, neural) COMET pass -- set to None to use the full test split.
MODELS = ["qwen", "mistral"]
MAX_COMET_SAMPLES = 200

## 0. Self-heal `src/cli/evaluate.py` before importing it

This environment has repeatedly picked up a corrupted copy of `src/cli/evaluate.py` where call arguments got wrapped in literal `<...>` (e.g. `fn(<a, b, c>)`), which is invalid Python. This cell detects and strips that pattern automatically, then verifies the file actually compiles, so the import below doesn't fail on a stale/corrupted copy. Safe to run even if the file is already clean — it's a no-op in that case.

In [ ]:
import re
import py_compile

evaluate_py_path = os.path.join("src", "cli", "evaluate.py")

if not os.path.exists(evaluate_py_path):
    print(f"⚠️ {evaluate_py_path} not found relative to {os.getcwd()} -- check working directory")
else:
    with open(evaluate_py_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Strips a stray '<' right after '(' and '>' right before ')' around a call's
    # argument list -- the exact shape of the corruption seen in this repo.
    repaired = re.sub(r"\(<([^<>()]*)>\)", r"(\1)", content)

    if repaired != content:
        with open(evaluate_py_path, "w", encoding="utf-8") as f:
            f.write(repaired)
        print(f"🔧 Repaired corrupted angle-bracket syntax in {evaluate_py_path}")
    else:
        print(f"✅ {evaluate_py_path} already clean, no repair needed")

    try:
        py_compile.compile(evaluate_py_path, doraise=True)
        print("✅ src/cli/evaluate.py compiles cleanly")
    except py_compile.PyCompileError as e:
        print(f"❌ src/cli/evaluate.py STILL has a syntax error after repair attempt -- open it manually:\n{e}")
        raise

## 1. SacreBLEU / chrF++ / Lexical Accuracy sweep

Reuses `evaluate_all_saved_checkpoints` from `src/cli/evaluate.py` — it already scans `checkpoints/{model_name}/` for every experiment with a saved best checkpoint, evaluates both translation directions, and evaluates the E0 zero-shot baseline. This cell just runs it once per model and tags the rows with which model produced them.

In [ ]:
import pandas as pd
from src.cli.evaluate import evaluate_all_saved_checkpoints

all_results = []
for model_name in MODELS:
    print(f"\n{'='*70}\nEvaluating all saved checkpoints for: {model_name}\n{'='*70}")
    df = evaluate_all_saved_checkpoints(model_name=model_name)
    df["Model"] = model_name
    all_results.append(df)

master_df = pd.concat(all_results, ignore_index=True)
master_df = master_df[["Model", "Experiment", "Direction", "SacreBLEU", "chrF++", "Lexical_Accuracy"]]
master_df

## 2. Save the combined sweep

In [ ]:
os.makedirs("outputs/evaluation_reports", exist_ok=True)
os.makedirs("data/results", exist_ok=True)

master_df.to_csv("outputs/evaluation_reports/master_evaluation_all_models.csv", index=False)
master_df.to_csv("data/results/master_evaluation_all_models.csv", index=False)
print("\u2705 Saved to outputs/evaluation_reports/master_evaluation_all_models.csv and data/results/master_evaluation_all_models.csv")

## 3. Neural COMET (English → Ekegusii, best checkpoint per experiment)

`evaluate_all_saved_checkpoints` only returns aggregate metrics, not raw predictions, and COMET needs (source, prediction, reference) triples — so this section regenerates predictions once per (model, experiment) specifically for COMET, capped to `MAX_COMET_SAMPLES` sentences to keep it tractable. Requires the `unbabel-comet` package (already in `requirements.txt`).

In [ ]:
from pathlib import Path
from src.experiments.base import BaseExperiment
from src.master_corpus.manager import MasterCorpusManager
from src.evaluation.comet import CometEvaluator
from src.models.qwen.inference import translate_with_qwen
from src.models.mistral.inference import translate_with_mistral

_TRANSLATE_FN = {"qwen": translate_with_qwen, "mistral": translate_with_mistral}


class _EvalHelper(BaseExperiment):
    experiment_id = "notebook14_comet"

    def build_training_tasks(self):
        raise NotImplementedError


helper = _EvalHelper(MasterCorpusManager())
test_pairs = helper.build_test_pairs("English", "Ekegusii")
if MAX_COMET_SAMPLES is not None and len(test_pairs) > MAX_COMET_SAMPLES:
    test_pairs = test_pairs.sample(MAX_COMET_SAMPLES, random_state=42)
sources = test_pairs["source"].tolist()
references = test_pairs["target"].tolist()

comet_evaluator = CometEvaluator()
comet_rows = []

for model_name in MODELS:
    checkpoints_base = Path(f"checkpoints/{model_name}")

    experiments_to_eval = [("E0_Baseline", None)]
    if checkpoints_base.exists():
        for exp_dir in sorted(checkpoints_base.iterdir()):
            if not exp_dir.is_dir():
                continue
            ckpts = sorted(
                [d for d in exp_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
                key=lambda x: int(x.name.split("-")[-1]),
            )
            if ckpts:
                experiments_to_eval.append((exp_dir.name, str(ckpts[-1])))

    for exp_name, adapter_path in experiments_to_eval:
        try:
            print(f"[{model_name}] {exp_name}: generating predictions for COMET...")
            predictions = _TRANSLATE_FN[model_name](sources, "English", "Ekegusii", adapter_path=adapter_path)
            result = comet_evaluator.compute(predictions, references, sources)
            comet_rows.append({"Model": model_name, "Experiment": exp_name, "COMET": round(result["mean_score"], 4)})
            print(f"  COMET: {result['mean_score']:.4f}")
        except Exception as e:
            print(f"  \u26a0\ufe0f skipped ({e})")

comet_df = pd.DataFrame(comet_rows)
comet_df

## 4. Merge COMET into the master table and re-save

In [ ]:
final_df = master_df.merge(comet_df, on=["Model", "Experiment"], how="left")
final_df.to_csv("outputs/evaluation_reports/master_evaluation_all_models.csv", index=False)
final_df.to_csv("data/results/master_evaluation_all_models.csv", index=False)
final_df

## 5. Quick comparison chart

In [ ]:
import matplotlib.pyplot as plt

plot_df = final_df[final_df["Direction"] == "English->Ekegusii"].copy()
pivot = plot_df.pivot(index="Experiment", columns="Model", values="SacreBLEU")

ax = pivot.plot(kind="bar", figsize=(12, 5))
ax.set_ylabel("SacreBLEU")
ax.set_title("SacreBLEU by Experiment and Model (English \u2192 Ekegusii)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
os.makedirs("outputs/figures", exist_ok=True)
plt.savefig("outputs/figures/full_evaluation_comparison.png", dpi=150)
plt.show()